<a href="https://colab.research.google.com/github/SATHRAMCHARAN/CSA6301--THREAT-INTELLIGENCE-AND-NETWORK-SECURITY/blob/main/exp32.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [29]:
# IDS Rules
IDS_RULES = [
    {
        "sid": 2000001,
        "msg": "Possible SQL Injection",
        "proto": "tcp",
        "dst_port": 80,
        "content": "union select"
    },
    {
        "sid": 2000002,
        "msg": "Directory Traversal Attempt",
        "proto": "tcp",
        "dst_port": 80,
        "content": "../../../etc/passwd"
    },
    {
        "sid": 2000003,
        "msg": "Suspicious RDP Brute Force Pattern",
        "proto": "tcp",
        "dst_port": 3389,
        "content": "login_attempt"
    }
]

# IDS Scan
def scan_packet(packet, rules=IDS_RULES):
    alerts = []
    payload_lower = packet["payload"].lower()

    for rule in rules:
        if (packet["proto"] == rule["proto"] and
                packet["dst_port"] == rule["dst_port"]):
            if rule["content"] in payload_lower:
                alerts.append(rule["msg"])

    return alerts

# IPS Processing
def ips_process(packet, rules, whitelist=None):
    """
    Decide whether to ALLOW or BLOCK a packet.
    """
    whitelist = whitelist or set()

    alerts = scan_packet(packet, rules)

    if not alerts:
        return {
            "action": "allow",
            "alerts": []
        }

    if packet.get("src_ip") in whitelist:
        return {
            "action": "allow",
            "alerts": alerts,
            "note": "Source whitelisted, alert suppressed from blocking"
        }

    return {
        "action": "block",
        "alerts": alerts
    }

# Sample packet
packet = {
    "src_ip": "192.168.1.100",
    "proto": "tcp",
    "dst_port": 80,
    "payload": "GET /index.php?id=1 UNION SELECT username,password FROM users"
}

# Whitelist
whitelist = {"10.0.0.1"}

# Process packet
result = ips_process(packet, IDS_RULES, whitelist)

print("Packet:")
print(packet)

print("\nIPS Result:")
print(result)

Packet:
{'src_ip': '192.168.1.100', 'proto': 'tcp', 'dst_port': 80, 'payload': 'GET /index.php?id=1 UNION SELECT username,password FROM users'}

IPS Result:
{'action': 'block', 'alerts': ['Possible SQL Injection']}
